In [83]:
import numpy as np

## Create a state vector

In [84]:
def create_state(vector_list):
    psi = np.array(vector_list, dtype=complex)
    return psi

### Normalization of the state

In [85]:
def normalize_state(psi):
    norm = np.sqrt(np.sum(np.abs(psi)**2))
    return psi/norm

In [86]:
def is_normalized(psi, tol=1e-10):
    norm = np.sum(np.abs(psi**2))
    return np.abs(norm-1)<tol

In [87]:
# example

psi = create_state([1,0])
psi = normalize_state(psi)

print("state vector:\n",psi)
print(is_normalized(psi))

state vector:
 [1.+0.j 0.+0.j]
True


## Constructing the density matrix

In [88]:
def density_matrix(psi):
    psi = psi.reshape(-1,1)   # Convert to column vector
    rho = psi @ psi.conj().T  # outer product @- matrix multiplication
    return rho

In [89]:
#test
rho = density_matrix(psi)
print("density_matrix:\n",rho)
print(np.trace(rho))      # trace should be equal to 1

density_matrix:
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
(1+0j)


In [90]:
# another example
psi1 = create_state([1,1])
psi1 = normalize_state(psi1)
rho1 = density_matrix(psi1)
print(rho1)
print(np.trace(rho1))      # trace should be equal to 1

[[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]
(0.9999999999999998+0j)


In [91]:
# Important properties
print(np.trace(rho))      # trace should be equal to 1
print(np.trace(rho @ rho))  # should be equal to 1 for pure states
print(np.allclose(rho,rho.conj().T))  # should be True for Hermitian matrices


(1+0j)
(1+0j)
True


## Tensor product for combine system


In [92]:
def tensor_product(A,B):
    return np.kron(A,B)

In [93]:
# example
psi_A = normalize_state(create_state([1,0]))    #|0>
psi_B = normalize_state(create_state([0,1]))    #|1>

psi_AB = tensor_product(psi_A,psi_B)
print(psi_AB)

[0.+0.j 1.+0.j 0.+0.j 0.+0.j]


In [94]:
# Tensor product for density matrix
rho_A = density_matrix(psi_A)
rho_B = density_matrix(psi_B)

rho_AB = tensor_product(rho_A,rho_B)
print(rho_AB)


[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]]


# Generalizing to many systems

In [95]:
def tensor_product_n(*args):
    result = args[0]
    for arr in args[1:]:
        result = np.kron(result,arr)
    return result    

In [96]:
# example
psi1 = normalize_state(create_state([1, 0]))
psi2 = normalize_state(create_state([0, 1]))
psi3 = normalize_state(create_state([1, 0]))

psi_total = tensor_product_n(psi1, psi2, psi3)

print(psi_total)

[0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]


## Reduced density matrix(partial trace)

In [97]:
# trace out system B and return density matrix for system A
def partial_trace_B(rho, dA,dB):
    rho_reshaped = rho.reshape(dA,dB,dA,dB)     # reshape into indices
    rho_A = np.zeros((dA,dA), dtype=complex)    # trace over B(axis 1 and 3)
    for j in range(dB):
        rho_A += rho_reshaped[:,j,:,j]          # this picks rho(i,j)(k,j)
    return rho_A    

In [112]:
# example bell state
psi_bell = normalize_state(create_state([1,0,0,1]))
print("psi_bell:\n",psi_bell)

rho_AB_bell = density_matrix(psi_bell)
print("rho_AB_bell:\n",rho_AB_bell)

# partial trace over B
rho_A_bell = partial_trace_B(rho_AB_bell, 2, 2)
print("partial trace over B:\n",rho_A_bell)

psi_bell:
 [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
rho_AB_bell:
 [[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]
partial trace over B:
 [[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


# General Partial trace for N systems

In [ ]:
def partial_trace(rho,dims,keep):    # rho- full density matrix, dims-subsystem dimension, keep-subsystem to keep
    dims = np.array(dims)            # convert dimensions to numpy array for easier indexing
    N= len(dims)                     # total number of subsystems


    # Subsystem to trace out
    trace_out = [i for i in range(N) if i not in keep]

    # reshape rho into tensor: rho shape= (d1,d2,...,dN,d1,d2,...,dN), after reshape= (d1,d2,...,dN,d1,d2,...,dN)
    reshaped = rho.reshape(*dims, *dims)

    # tracing out unwanted systems
    for i in sorted(trace_out,  reverse = True):
        reshaped= np.trace(reshaped, axis1 = i, axis2 = i+N)

    # final reshape
    dims_keep = dims[keep]
    final_dim = int(np.prod(dims_keep))

    return reshaped.reshape(final_dim,final_dim)
    


In [114]:
# example bell state using generalised function
psi_bell = normalize_state(create_state([1,0,0,1]))

rho_AB_bell = density_matrix(psi_bell)

# partial trace over B
rho_A_bell = partial_trace(rho_AB_bell, dims=[2, 2], keep=[0])
print("partial trace over B:\n",rho_A_bell)

partial trace over B:
 [[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


In [115]:

rho_B_bell = partial_trace(rho_AB_bell, dims=[2, 2], keep=[0])
print(rho_B_bell)

[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


### 3:3 system example

In [116]:
psi1 = normalize_state(create_state([1,0]))
psi2 = normalize_state(create_state([0,1]))
psi3 = normalize_state(create_state([1,0]))

psi_total = tensor_product_n(psi1,psi2,psi3)
rho = density_matrix(psi_total)

# keep only system 0
rho_reduced = partial_trace(rho,dims=[2,2,2],keep=[1])
print(rho_reduced)


[[0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
